In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from pathlib import Path

# Embedding output directory. UPDATE for your use case.
emb_output_dir = '/raid/embeddings/BeIR/trec-news-generated-queries/fp32_768d_clean'

# Load huggingface dataset
ds = load_dataset("BeIR/trec-news-generated-queries")
num_rows = ds['train'].num_rows
print(f'Number of rows: {num_rows}')

print(ds['train'].features)

# Data Cleaning

In [ ]:
clean_data = ds['train'].to_pandas()

# Remove short "text" fields
clean_data['text_len'] = clean_data['text'].str.len()
clean_data = clean_data[clean_data['text_len'] > 20]

# Remove duplicate data based on _id
clean_data = clean_data[['_id', 'text']].drop_duplicates()

print(f'Duplicate data fraction: {1 - len(clean_data)/num_rows: 0.2}')

# Embed Text Passages

In [ ]:
# Download from the 🤗 Hub. May need to create a token and "acknowledge license" on HF.
model = SentenceTransformer("google/embeddinggemma-300m") #, token='hf_xxx')

# Start a pool with all available GPUs
pool = model.start_multi_process_pool()

# Encode with batch_size to optimize speed
document_embeddings = model.encode_document(clean_data['text'], pool=pool,
                                            batch_size=128, show_progress_bar=True)

# Optional: Stop the pool when finished
model.stop_multi_process_pool(pool)

print(document_embeddings.shape)

In [ ]:
# Write data columns as individual numpy files per column in same format
# as processed miracl dataset.

# Create directory and parents if they don't exist
Path(emb_output_dir).mkdir(parents=True, exist_ok=True)

np.save(os.path.join(emb_output_dir, "id.npy"), np.array(clean_data['_id'], dtype='U36'))
np.save(os.path.join(emb_output_dir, "title.npy"), np.array(clean_data['title'], dtype='U512'))
np.save(os.path.join(emb_output_dir, "text.npy"), np.array(clean_data['text'], dtype='U32768'))
np.save(os.path.join(emb_output_dir, "embedding.npy"), document_embeddings)